# Initial Data Audit — Retail Sales Dataset
Notebook audit dataset yang dipilih pada Week 2 (Data Collection).

**Tujuan:** memastikan data siap digunakan untuk analisis dengan mengecek jumlah data, jumlah kolom, missing value, duplikat, tipe data, dan permasalahan lain.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('retail_sales_dataset.csv')
df.head()

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
0,1,2023-11-24,CUST001,Male,34,Beauty,3,50,150
1,2,2023-02-27,CUST002,Female,26,Clothing,2,500,1000
2,3,2023-01-13,CUST003,Male,50,Electronics,1,30,30
3,4,2023-05-21,CUST004,Male,37,Clothing,1,500,500
4,5,2023-05-06,CUST005,Male,30,Beauty,2,50,100


## 1. Info Umum Dataset

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    1000 non-null   int64
 1   Date              1000 non-null   str  
 2   Customer ID       1000 non-null   str  
 3   Gender            1000 non-null   str  
 4   Age               1000 non-null   int64
 5   Product Category  1000 non-null   str  
 6   Quantity          1000 non-null   int64
 7   Price per Unit    1000 non-null   int64
 8   Total Amount      1000 non-null   int64
dtypes: int64(5), str(4)
memory usage: 70.4 KB


## 2. Jumlah Data & Jumlah Kolom

In [3]:
print('Jumlah baris :', df.shape[0])
print('Jumlah kolom :', df.shape[1])

Jumlah baris : 1000
Jumlah kolom : 9


## 3. Missing Value per Kolom

In [4]:
df.isnull().sum()

Transaction ID      0
Date                0
Customer ID         0
Gender              0
Age                 0
Product Category    0
Quantity            0
Price per Unit      0
Total Amount        0
dtype: int64

## 4. Data Duplikat

In [5]:
print('Jumlah baris duplikat:', df.duplicated().sum())

Jumlah baris duplikat: 0


## 5. Tipe Data per Kolom

In [6]:
df.dtypes

Transaction ID      int64
Date                  str
Customer ID           str
Gender                str
Age                 int64
Product Category      str
Quantity            int64
Price per Unit      int64
Total Amount        int64
dtype: object

## 6. Statistik Deskriptif

In [7]:
df.describe(include='all')

,Transaction ID,Date,Customer ID,Gender,Age,Product Category,Quantity,Price per Unit,Total Amount
count,1000.000000,1000,1000,1000,1000.00000,1000,1000.000000,1000.000000,1000.000000
unique,NaN,345,1000,2,NaN,3,NaN,NaN,NaN
top,NaN,2023-05-16,CUST001,Female,NaN,Clothing,NaN,NaN,NaN
freq,NaN,11,1,510,NaN,351,NaN,NaN,NaN
mean,500.500000,NaN,NaN,NaN,41.39200,NaN,2.514000,179.890000,456.000000
std,288.819436,NaN,NaN,NaN,13.68143,NaN,1.132734,189.681356,559.997632
min,1.000000,NaN,NaN,NaN,18.00000,NaN,1.000000,25.000000,25.000000
25%,250.750000,NaN,NaN,NaN,29.00000,NaN,1.000000,30.000000,60.000000
50%,500.500000,NaN,NaN,NaN,42.00000,NaN,3.000000,50.000000,135.000000
75%,750.250000,NaN,NaN,NaN,53.00000,NaN,4.000000,300.000000,900.000000


## 7. Pengecekan Nilai Kategorikal (Typo / Inkonsistensi)

In [8]:
print('Gender          :', df['Gender'].unique())
print('Product Category:', df['Product Category'].unique())

Gender          : <StringArray>
['Male', 'Female']
Length: 2, dtype: str
Product Category: <StringArray>
['Beauty', 'Clothing', 'Electronics']
Length: 3, dtype: str


## 8. Pengecekan Format Customer ID

In [9]:
# Pola yang diharapkan: CUST + 3 digit angka, misal CUST001
mask_invalid = ~df['Customer ID'].str.match(r'^CUST\d{3}$')
print('Jumlah Customer ID yang TIDAK sesuai pola CUSTxxx (3 digit):', mask_invalid.sum())
df.loc[mask_invalid, ['Transaction ID', 'Customer ID']]

Jumlah Customer ID yang TIDAK sesuai pola CUSTxxx (3 digit): 1


,Transaction ID,Customer ID
999,1000,CUST1000


## 9. Pengecekan Nilai Negatif / Nol

In [10]:
for col in ['Age', 'Quantity', 'Price per Unit', 'Total Amount']:
    n_invalid = (df[col] <= 0).sum()
    print(f'{col}: min={df[col].min()}, max={df[col].max()}, nilai <= 0: {n_invalid}')

Age: min=18, max=64, nilai <= 0: 0
Quantity: min=1, max=4, nilai <= 0: 0
Price per Unit: min=25, max=500, nilai <= 0: 0
Total Amount: min=25, max=2000, nilai <= 0: 0


## 10. Pengecekan Outlier (Metode IQR)

In [11]:
for col in ['Age', 'Quantity', 'Price per Unit', 'Total Amount']:
    q1, q3 = df[col].quantile([0.25, 0.75])
    iqr = q3 - q1
    low, high = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    outliers = df[(df[col] < low) | (df[col] > high)]
    print(f'{col}: {len(outliers)} outlier (batas normal: {low:.1f} - {high:.1f})')

Age: 0 outlier (batas normal: -7.0 - 89.0)
Quantity: 0 outlier (batas normal: -3.5 - 8.5)
Price per Unit: 0 outlier (batas normal: -375.0 - 705.0)
Total Amount: 0 outlier (batas normal: -1200.0 - 2160.0)


## 11. Validasi Rumus: Total Amount = Quantity x Price per Unit

In [12]:
mismatch = df[df['Total Amount'] != df['Quantity'] * df['Price per Unit']]
print('Jumlah baris yang rumusnya tidak konsisten:', len(mismatch))

Jumlah baris yang rumusnya tidak konsisten: 0


## 12. Kesimpulan Audit

| Pengecekan | Hasil |
|---|---|
| Jumlah data | 1.000 baris |
| Jumlah kolom | 9 kolom |
| Missing value | 0 (semua kolom) |
| Data duplikat | 0 |
| Nilai negatif/nol | Tidak ditemukan |
| Outlier (IQR) | Tidak ditemukan |
| Konsistensi rumus Total Amount | 100% konsisten |
| Inkonsistensi format | **Ditemukan**: 1 Customer ID (`CUST1000`) tidak mengikuti pola 3 digit standar |
| Tipe data kolom Date | Masih string, perlu dikonversi ke datetime sebelum analisis waktu |

**Status akhir:** dataset secara umum sudah bersih dan siap digunakan, dengan 1 catatan minor (inkonsistensi format Customer ID) yang perlu distandardisasi pada tahap data cleaning berikutnya.